In [3]:

import pyspark
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [8]:
import requests
from tqdm import tqdm  # uv add tqdm for progress bar

url = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz"
response = requests.get(url, stream=True)
response.raise_for_status()

total_size = int(response.headers.get('content-length', 0))
with open('fhvhv_tripdata_2021-01.csv.gz', 'wb') as file, tqdm(
    desc="Downloading", total=total_size, unit='B', unit_scale=True, unit_divisor=1024
) as pbar:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            file.write(chunk)
            pbar.update(len(chunk))

print("\n✅ Download complete!")


Downloading: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 124M/124M [00:07<00:00, 18.1MB/s]


✅ Download complete!


In [9]:
import os
size_gb = os.path.getsize('fhvhv_tripdata_2021-01.csv.gz') / (1024**3)
print(f"✅ File size: {size_gb:.2f} GB (should be ~1.03 GB)")

# Decompress
import gzip, shutil
with gzip.open('fhvhv_tripdata_2021-01.csv.gz', 'rb') as f_in:
    with open('fhvhv_tripdata_2021-01.csv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
print("✅ Ready for Spark!")


✅ File size: 0.12 GB (should be ~1.03 GB)
✅ Ready for Spark!


In [10]:
!wc -l fhvhv_tripdata_2021-01.csv

11908469 fhvhv_tripdata_2021-01.csv


In [5]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [16]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [17]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [18]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [19]:
!wc -l head.csv

1001 head.csv


In [21]:
import pandas as pd

In [22]:
df_pandas = pd.read_csv('head.csv')

In [23]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [24]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [6]:
from pyspark.sql import types

In [7]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [8]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [6]:
df = df.repartition(24)

In [5]:
import os
os.environ['HADOOP_HOME'] = r'C:\hadoop'

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('DE Zoomcamp') \
    .config('spark.sql.adaptive.enabled', 'false') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'false') \
    .config('spark.hadoop.fs.file.impl', 'org.apache.hadoop.fs.RawLocalFileSystem') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .getOrCreate()


In [ ]:
df.write.parquet('fhvhv/2021/01/')

In [ ]:
df = spark.read.parquet('fhvhv/2021/01/')

In [ ]:
df.printSchema()

In [4]:
import shutil
shutil.rmtree('fhvhv', ignore_errors=True)
